In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## project directories
data_dir = Path("../data")
geo_dir = Path("../geo_data")
geo_dir.mkdir(parents=True, exist_ok=True)

# load our regional dataset

regional = pd.read_csv(
    data_dir / "measles_region_year.csv"
)

# Basic data validation
print("Dataset shape:", regional.shape)
print("Number of health regions:",
      regional["health_region"].nunique())
print("Years:", regional["year"].min(),
      "-", regional["year"].max())

print("Total reported cases:",
      regional["reported_cases"].sum())

print("\nRegion names:")
print(sorted(regional["health_region"].unique()))

Dataset shape: (200, 7)
Number of health regions: 20
Years: 2014 - 2023
Total reported cases: 5997

Region names:
['Al-Ahsa', 'Al-Bahah', 'Al-Jouf', 'Aseer', 'Bishah', 'Eastern', 'Ha`il', 'Hafr Al-Baten', 'Jazan', 'Jeddah', 'Makkah', 'Medinah', 'Najran', 'Northern', 'Qaseem', 'Qunfudah', 'Qurayyat', 'Riyadh', 'Ta`if', 'Tabouk']


In [2]:
import requests
service_url=(
    "https://gpcict.sa/server/rest/services/"
    "GeoSA_Prod_EN/Hospitals/MapServer"
)

for layer_id in [11 ,12]:

    response = requests.get(
        f"{service_url}/{layer_id}",
        params={"f": "json"},
        timeout=30
    )

    response.raise_for_status()
    layer = response.json()

    if "error" in layer:
        print(f"Layer {layer_id}: {layer['error']}")
        continue

    print(f"\n{'=' * 45}")
    print(f"LAYER {layer_id}")
    print(f"{'=' * 45}")

    print("Name:", layer.get("name"))
    print("Geometry:", layer.get("geometryType"))

    print("\nAvailable fields:")
    for field in layer.get("fields", []):
        print("-", field["name"])



LAYER 11
Name: Hospitals By Region
Geometry: esriGeometryPolygon

Available fields:
- SHAPE.AREA
- SHAPE.LEN
- OBJECTID
- REGIONAR
- CITYAR
- DESCRIPTIONAR
- NOOFHOSPITALSEN
- NOOFBEDSEN
- NOOFHOSPITALSAR
- OBJECTID_1
- المنطقة_الصحية
- NOOFBEDSAR
- NOOFDOCTORSAR
- NOOFNURSESAR
- REGIONEN
- DESCRIPTIONEN
- NOOFDOCTORSEN
- NOOFNURSESEN
- CITYEN
- SHAPE

LAYER 12
Name: Hospitals By Region
Geometry: esriGeometryPolygon

Available fields:
- SHAPE.AREA
- SHAPE.LEN
- OBJECTID
- REGIONAR
- CITYAR
- DESCRIPTIONAR
- NOOFHOSPITALSEN
- NOOFBEDSEN
- NOOFHOSPITALSAR
- OBJECTID_1
- المنطقة_الصحية
- NOOFBEDSAR
- NOOFDOCTORSAR
- NOOFNURSESAR
- REGIONEN
- DESCRIPTIONEN
- NOOFDOCTORSEN
- NOOFNURSESEN
- CITYEN
- SHAPE


In [3]:

for layer_id in [11, 12]:

    query_url = f"{service_url}/{layer_id}/query"

    response = requests.get(
        query_url,
        params={
            "where": "1=1",
            "outFields": "REGIONEN,CITYEN",
            "returnGeometry": "false",
            "f": "json"
        },
        timeout=30
    )

    response.raise_for_status()
    result = response.json()

    if "error" in result:
        print(f"Layer {layer_id}:", result["error"])
        continue

    records = [
        feature["attributes"]
        for feature in result.get("features", [])
    ]

    geo_names = pd.DataFrame(records)

    print(f"\nLAYER {layer_id}")
    print("Number of features:", len(geo_names))

    if "REGIONEN" in geo_names.columns:
        print("Unique health-region names:")
        print(sorted(
            geo_names["REGIONEN"]
            .dropna()
            .astype(str)
            .unique()
        ))

    display(geo_names.head())



LAYER 11
Number of features: 13
Unique health-region names:
['Al- Bahah', 'Al- Jouf', 'Aseer', 'Eastern', "Ha'il", 'Jazan', 'Medinah', 'Najran', 'Northern', 'Qaseem', 'Riyadh', 'Tabouk', 'The Holy Capital']


,REGIONEN,CITYEN
0,Qaseem,Buraydah
1,Al- Bahah,Al-Baha
2,Northern,Arar
3,Al- Jouf,Sakaka
4,Aseer,Abha



LAYER 12
Number of features: 13
Unique health-region names:
['Al- Bahah', 'Al- Jouf', 'Aseer', 'Eastern', "Ha'il", 'Jazan', 'Medinah', 'Najran', 'Northern', 'Qaseem', 'Riyadh', 'Tabouk', 'The Holy Capital']


,REGIONEN,CITYEN
0,Qaseem,Buraydah
1,Al- Bahah,Al-Baha
2,Northern,Arar
3,Al- Jouf,Sakaka
4,Aseer,Abha


In [4]:

# Preliminary province-to-health-region reference
province_groups = {
    "Makkah": [
        "Makkah", "Jeddah", "Ta`if", "Qunfudah"
    ],
    "Eastern": [
        "Eastern", "Al-Ahsa", "Hafr Al-Baten"
    ],
    "Aseer": [
        "Aseer", "Bishah"
    ],
    "Al-Jouf": [
        "Al-Jouf", "Qurayyat"
    ],
    "Riyadh": ["Riyadh"],
    "Medinah": ["Medinah"],
    "Qaseem": ["Qaseem"],
    "Tabouk": ["Tabouk"],
    "Ha`il": ["Ha`il"],
    "Northern": ["Northern"],
    "Jazan": ["Jazan"],
    "Najran": ["Najran"],
    "Al-Bahah": ["Al-Bahah"]
}

crosswalk = pd.DataFrame(
    [
        {"administrative_province": province,
         "health_region": health_region}
        for province, regions in province_groups.items()
        for health_region in regions
    ]
)

expected = set(regional["health_region"].unique())
mapped = set(crosswalk["health_region"])

print("Administrative provinces:",
      crosswalk["administrative_province"].nunique())
print("Mapped health regions:", len(mapped))
print("Unmatched:", sorted(expected - mapped))
print("Unexpected:", sorted(mapped - expected))
print("Duplicate health regions:",
      crosswalk["health_region"].duplicated().sum())

display(crosswalk.sort_values(
    ["administrative_province", "health_region"]
))

crosswalk.to_csv(
    geo_dir / "preliminary_health_region_province_crosswalk.csv",
    index=False
)


Administrative provinces: 13
Mapped health regions: 20
Unmatched: []
Unexpected: []
Duplicate health regions: 0


,administrative_province,health_region
19,Al-Bahah,Al-Bahah
9,Al-Jouf,Al-Jouf
10,Al-Jouf,Qurayyat
7,Aseer,Aseer
8,Aseer,Bishah
5,Eastern,Al-Ahsa
4,Eastern,Eastern
6,Eastern,Hafr Al-Baten
15,Ha`il,Ha`il
17,Jazan,Jazan


In [6]:

from pathlib import Path

geo_dir = Path("../geo_data")
geo_dir.mkdir(parents=True, exist_ok=True)

print("Geographic files available:")

for file in sorted(geo_dir.iterdir()):
    if file.is_file():
        print("-", file.name)


Geographic files available:
- .DS_Store
- preliminary_health_region_province_crosswalk.csv


In [7]:

import geopandas as gpd
from pathlib import Path

# Correct location based on your screenshot
geojson_path = Path(
    "../geo_data/geojson/ksa_governorates.geojson"
)

# Read geographic boundaries
governorates = gpd.read_file(geojson_path)

print("Total geographic features:", len(governorates))
print("Coordinate reference system:", governorates.crs)

print("\nAvailable columns:")
print(governorates.columns.tolist())

print("\nFirst five records:")
display(governorates.head())


ModuleNotFoundError: No module named 'geopandas'